# R15 · Segmentación de clientes con K-means

**Proyecto Final Integrador · Machine Learning I · Universidad EAN**
Integrantes: Claudia Naranjo, Yery Pedraza y Nicolás Ruiz.

Componente A: tipos de cliente según cómo pagan (anteproyecto §4, A), con el diseño de D-31. Se
resume en [`segmentacion_R15.md`](segmentacion_R15.md).

| Sección | Criterio de R15 |
|---|---|
| §1 Perfil y columnas | ≥ 4 variables justificadas; columnas excluidas con su motivo |
| §2 Partición y preparación | Partición antes; recorte y escala aprendidos solo en entrenamiento |
| §3 Sin preparar frente a preparado | Tamaños de los grupos y qué cambió |
| §4 Inercia y silueta | k de 2 a 10; ¿hay codo? |
| §5 Elección de k y semillas | Justificación que no depende solo de un índice; 5 semillas |
| §6 Perfil de los grupos | Medianas, tamaño y nombre |
| §7 Sensibilidad al mínimo de facturas | D-09: mínimos de 3 y 10 |

**Confidencialidad:** solo cifras agregadas; nunca se muestra un cliente.

## 0. Configuración

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos_crudos").is_dir())
sys.path.insert(0, str(RAIZ / "src"))

from cartera import datos
from cartera import segmentacion as sg
from cartera.config import SEMILLA

np.random.seed(SEMILLA)
print("Semilla", SEMILLA, "| Python", sys.version.split()[0], "| pandas", pd.__version__,
      "| numpy", np.__version__, "| scikit-learn", sklearn.__version__)

Semilla 2026 | Python 3.14.7 | pandas 3.0.6 | numpy 2.5.3 | scikit-learn 1.9.1


## 1. Perfil de cada cliente y columnas

**Unidad:** el cliente (`nit_cliente`) con al menos 5 facturas pagadas (D-09). **Ventana:** facturas
emitidas de 2022-01 a 2025-06 con desenlace conocido al 1/07/2025. Lo posterior se reserva como
desenlace externo para R17.

| Variable | Qué mide | Por qué entra (anteproyecto §4, A) |
|---|---|---|
| `atraso_mediano` | Atraso típico de sus facturas pagadas (D-16, D-20) | Qué tan tarde suele pagar |
| `atraso_iqr` | Rango intercuartílico del atraso | Qué tan variable es ese atraso |
| `pct_tardias` | % de facturas con más de 30 días de atraso | Proporción de pagos tardíos |
| `facturas_por_anio` | Facturas emitidas por año en la ventana | Con qué frecuencia se le factura |
| `log_monto_mediano` | Log del monto inicial mediano | Cuánto se le factura |
| `pct_varios_abonos` | % de facturas pagadas con más de un RC | Si suele pagar con abonos parciales |
| `pct_cierre_nota` | % de facturas cerradas con notas o ajustes | Cierres sin pago (D-16) |

**Excluidas y su motivo:**
- `nit_cliente`, `cliente`, `id_*`: identificadores; el algoritmo no debe distinguir clientes por su
  código.
- `empresa`, `nombre_vendedor`: el grupo se trata como una sola empresa (D-27) y no se juzga a los
  vendedores (D-02).
- Todo lo emitido o cerrado desde el 1/07/2025: es el desenlace externo de R17.
- El puntaje del modelo B: la segmentación debe describir a los clientes por sí misma.

In [2]:
saldos, movs = datos.leer_todo()
perfil = sg.perfil(saldos, movs)
print("Clientes con al menos 5 facturas pagadas:", len(perfil))
perfil[sg.VARIABLES].describe(percentiles=[.25, .5, .75]).T.round(1)

Clientes con al menos 5 facturas pagadas: 226


,count,mean,std,min,25%,50%,75%,max
atraso_mediano,226.0,24.4,33.8,-25.5,4.0,15.8,33.0,277.0
atraso_iqr,226.0,28.0,33.0,1.0,9.5,18.0,31.4,239.0
pct_tardias,226.0,32.1,31.3,0.0,5.1,20.6,54.1,100.0
facturas_por_anio,226.0,15.5,11.5,1.4,8.7,13.2,17.5,83.9
log_monto_mediano,226.0,15.1,0.8,12.6,14.6,15.0,15.5,18.5
pct_varios_abonos,226.0,4.0,6.7,0.0,0.0,0.0,6.1,57.1
pct_cierre_nota,226.0,11.2,13.2,0.0,3.2,7.4,14.4,75.0


## 2. Partición y preparación

Los clientes se separan al azar **antes** de cualquier transformación: 70 % para entrenamiento y
30 % para prueba (semilla 2026). El recorte de extremos (percentiles 1 y 99) y la escala estándar se
aprenden solo con el entrenamiento, dentro de un `Pipeline`, y la prueba solo se transforma.

In [3]:
ent, prueba = sg.particion(perfil, SEMILLA)
assert not set(ent.index) & set(prueba.index)
prep = sg.preparacion().fit(ent[sg.VARIABLES])
Z = prep.transform(ent[sg.VARIABLES])
print("Entrenamiento:", len(ent), "clientes | prueba:", len(prueba), "clientes")
pd.DataFrame({"límite inferior (p1)": prep.named_steps["recorte"].lim_inf_,
              "límite superior (p99)": prep.named_steps["recorte"].lim_sup_}, index=sg.VARIABLES).round(1)

Entrenamiento: 158 clientes | prueba: 68 clientes


,límite inferior (p1),límite superior (p99)
atraso_mediano,-13.2,153.9
atraso_iqr,2.1,167.0
pct_tardias,0.0,100.0
facturas_por_anio,1.6,60.9
log_monto_mediano,13.6,17.4
pct_varios_abonos,0.0,25.9
pct_cierre_nota,0.0,66.1


## 3. K-means sin preparar frente a preparado

Mismo k (el elegido en §5, k = 2) y misma semilla. Sin preparar, K-means mide distancias en las
unidades originales: domina la variable con números más grandes.

In [4]:
from sklearn.cluster import KMeans

K = sg.K   # k = 2, elegido en §5 (D-32)
sin_preparar = KMeans(K, n_init=sg.N_INIT, random_state=SEMILLA).fit(ent[sg.VARIABLES])
preparado = KMeans(K, n_init=sg.N_INIT, random_state=SEMILLA).fit(Z)
pd.DataFrame({"sin preparar": np.sort(np.bincount(sin_preparar.labels_))[::-1],
              "preparado": np.sort(np.bincount(preparado.labels_))[::-1]},
             index=[f"grupo {i + 1}" for i in range(K)])

,sin preparar,preparado
grupo 1,123,120
grupo 2,35,38


In [5]:
pd.Series(ent[sg.VARIABLES].std(), name="desviación estándar en unidades originales").round(1).to_frame()

,desviación estándar en unidades originales
atraso_mediano,34.3
atraso_iqr,31.9
pct_tardias,31.6
facturas_por_anio,12.1
log_monto_mediano,0.8
pct_varios_abonos,7.0
pct_cierre_nota,13.3


## 4. Inercia y silueta para k de 2 a 10

In [6]:
from sklearn.metrics import adjusted_rand_score, silhouette_score

filas = []
for k in range(2, 11):
    km = KMeans(k, n_init=sg.N_INIT, random_state=SEMILLA).fit(Z)
    tam = np.bincount(km.labels_)
    filas.append({"k": k, "inercia": km.inertia_, "caída de inercia": np.nan,
                  "silueta": silhouette_score(Z, km.labels_),
                  "grupo más chico": tam.min(), "% del grupo más chico": tam.min() / len(Z) * 100})
indices = pd.DataFrame(filas).set_index("k")
indices["caída de inercia"] = -indices.inercia.diff()
indices.round(3)

,inercia,caída de inercia,silueta,grupo más chico,% del grupo más chico
k,,,,,
2,812.620,NaN,0.353,38,24.051
3,710.818,101.802,0.259,11,6.962
4,621.097,89.720,0.218,10,6.329
5,539.933,81.164,0.231,10,6.329
6,474.832,65.101,0.251,9,5.696
7,421.018,53.814,0.231,9,5.696
8,391.719,29.299,0.235,4,2.532
9,366.702,25.017,0.223,4,2.532
10,345.513,21.190,0.232,3,1.899


## 5. Elección de k y estabilidad entre semillas

**Regla (D-31):** se consideran los k en que ningún grupo tiene menos del 5 % de los clientes y cada
grupo se puede nombrar con un tratamiento de cartera. Entre ellos, se mira la silueta, la caída de
inercia y la estabilidad. La estabilidad se mide con 5 semillas: el índice de Rand ajustado (ARI)
entre las particiones de cada par de semillas (1 = idénticas). Se reporta con 10 reinicios de
K-means (el valor habitual) y con 50, el que se usa.

In [7]:
semillas = [SEMILLA + i for i in range(5)]
filas = []
for n_init in [10, sg.N_INIT]:
    for k in [2, 3, 4, 5]:
        modelos = [KMeans(k, n_init=n_init, random_state=s).fit(Z) for s in semillas]
        aris = [adjusted_rand_score(modelos[i].labels_, modelos[j].labels_) for i in range(5) for j in range(i + 1, 5)]
        filas.append({"reinicios": n_init, "k": k, "ARI mínimo entre semillas": min(aris), "ARI medio": np.mean(aris),
                      "inercia mínima": min(m.inertia_ for m in modelos), "inercia máxima": max(m.inertia_ for m in modelos)})
pd.DataFrame(filas).set_index(["reinicios", "k"]).round(3)

ARI mínimo entre semillas  ARI medio  inercia mínima  \
reinicios k                                                         
10        2                      1.000      1.000         812.620   
          3                      0.234      0.500         710.818   
          4                      0.719      0.822         620.045   
          5                      0.542      0.683         539.446   
50        2                      1.000      1.000         812.620   
          3                      0.969      0.983         710.681   
          4                      0.767      0.859         620.016   
          5                      0.853      0.911         539.446   

             inercia máxima  
reinicios k                  
10        2         812.620  
          3         717.663  
          4         622.791  
          5         548.585  
50        2         812.620  
          3         711.265  
          4         622.462  
          5         539.933

**Elección: k = 2 (D-32).** No hay un codo claro: la inercia baja de forma pareja. Con 10 reinicios,
k = 3 cambia según la semilla; con 50 es estable. Pero la prueba decisiva es la de §7: al cambiar el
mínimo de facturas pagadas de 5 a 3 o a 10, solo k = 2 mantiene los grupos (ARI ≥ 0,80); los de
k = 3 y k = 4 se reorganizan. k = 2 además tiene la mejor silueta, ningún grupo por debajo del 5 % y
dos grupos que se nombran con un tratamiento. Se descartó k = 3, aunque encajaba con los tres
tratamientos del anteproyecto, porque su tercer grupo (11 clientes muy tardíos) desaparece con
un cambio de un criterio arbitrario del equipo. R16 revisa si otro algoritmo los encuentra.

## 6. Perfil de los grupos (k = 2)

Medianas de cada variable en unidades originales y tamaño de cada grupo, en el entrenamiento.

In [8]:
modelo = KMeans(K, n_init=sg.N_INIT, random_state=SEMILLA).fit(Z)
ent_g = ent.assign(grupo=modelo.labels_)
perfil_grupos = ent_g.groupby("grupo")[sg.VARIABLES].median().round(1)
perfil_grupos["clientes"] = ent_g.grupo.value_counts().sort_index()
perfil_grupos["% de clientes"] = (perfil_grupos.clientes / len(ent_g) * 100).round(1)
perfil_grupos.sort_values("atraso_mediano")

,atraso_mediano,atraso_iqr,pct_tardias,facturas_por_anio,log_monto_mediano,pct_varios_abonos,pct_cierre_nota,clientes,% de clientes
grupo,,,,,,,,,
0,9.0,14.0,10.4,13.1,14.9,0.0,7.1,120,75.9
1,51.5,50.0,81.3,12.7,15.2,2.4,11.8,38,24.1


In [9]:
# Los clientes de prueba se asignan al centro más cercano, con la preparación del entrenamiento
prueba_g = prueba.assign(grupo=modelo.predict(prep.transform(prueba[sg.VARIABLES])))
pd.DataFrame({"entrenamiento": ent_g.grupo.value_counts(normalize=True).sort_index() * 100,
              "prueba": prueba_g.grupo.value_counts(normalize=True).sort_index() * 100}).round(1).rename_axis("grupo")

,entrenamiento,prueba
grupo,,
0,75.9,75.0
1,24.1,25.0


## 7. Sensibilidad al mínimo de facturas pagadas (D-09)

Se repite todo (partición, preparación y K-means) con mínimos de 3 y de 10 facturas, para k = 2, 3 y 4. Para
los clientes que están en las dos versiones, se compara su grupo con el ARI. Como los números de
grupo son arbitrarios, el ARI es la medida correcta: no depende de cómo se numeren.

In [10]:
def segmentar(p, k):
    e, _ = sg.particion(p, SEMILLA)
    pr = sg.preparacion().fit(e[sg.VARIABLES])
    km = KMeans(k, n_init=sg.N_INIT, random_state=SEMILLA).fit(pr.transform(e[sg.VARIABLES]))
    return pr, km

perfiles = {m: sg.perfil(saldos, movs, min_pagadas=m) for m in [3, 10]}
filas = []
for k in [2, 3, 4]:
    base_prep, base_km = segmentar(perfil, k)
    for minimo, p in perfiles.items():
        pr, km = segmentar(p, k)
        comunes = p.index.intersection(perfil.index)
        a = base_km.predict(base_prep.transform(perfil.loc[comunes, sg.VARIABLES]))
        b = km.predict(pr.transform(p.loc[comunes, sg.VARIABLES]))
        filas.append({"k": k, "mínimo de facturas pagadas": minimo, "clientes": len(p),
                      "clientes en común con el mínimo 5": len(comunes),
                      "ARI frente al mínimo 5": adjusted_rand_score(a, b)})
pd.DataFrame(filas).set_index(["k", "mínimo de facturas pagadas"]).round(3)

clientes  clientes en común con el mínimo 5  \
k mínimo de facturas pagadas                                                
2 3                                281                                226   
  10                               191                                191   
3 3                                281                                226   
  10                               191                                191   
4 3                                281                                226   
  10                               191                                191   

                              ARI frente al mínimo 5  
k mínimo de facturas pagadas                          
2 3                                            0.797  
  10                                           0.845  
3 3                                            0.280  
  10                                           0.349  
4 3                                            0.336  
  10                                           0.534

**Nombres y tratamiento propuestos.**
- **Paga a tiempo o casi:** atraso mediano de pocos días y pocas facturas tardías. Le basta el correo
  semanal.
- **Paga tarde:** atraso mediano de más de un mes y la mayoría de sus facturas tardías. Requiere
  llamada.

**Lo que el agrupamiento no prueba.** Los grupos describen cómo pagó cada cliente entre 2022 y
mediados de 2025; no dicen por qué paga así ni que vaya a seguir igual. Que un cliente esté en
«paga tarde» no prueba que una llamada lo vaya a cambiar: eso solo lo mide un piloto (anteproyecto
§5, C). R17 mide si los grupos anticipan el comportamiento posterior.